# Q2.2 Hyperparameter Sweep
Run sweep from this notebook and inspect with W&B API.

In [1]:
import sys
from pathlib import Path
import argparse
import numpy as np
import wandb
from sklearn.metrics import precision_recall_fscore_support

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.data_loader import load_dataset
from src.ann.neural_network import NeuralNetwork

ENTITY = 'anandhakrishnanm21-indian-institute-of-technology-madras'


In [2]:
def to_namespace(cfg):
    d = dict(cfg)
    defaults = {
        'hidden_size': [128, 64],
        'activation': 'relu',
        'weight_init': 'xavier',
        'loss': 'cross_entropy',
        'learning_rate': 1e-3,
        'weight_decay': 0.0,
        'optimizer': 'sgd',
        'epochs': 10,
        'batch_size': 64,
    }
    defaults.update(d)
    defaults['num_layers'] = len(defaults['hidden_size'])
    return argparse.Namespace(**defaults)


def evaluate_split(model, X, Y):
    logits = model.forward(X)
    loss, acc = model.evaluate(X, Y)
    y_true = np.argmax(Y, axis=1)
    y_pred = np.argmax(logits, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    return {
        'loss': float(loss),
        'acc': float(acc),
        'precision': float(p),
        'recall': float(r),
        'f1': float(f1),
        'y_true': y_true,
        'y_pred': y_pred,
    }


def train_one_run(cfg, project, run_name, dataset_name='mnist', extras_fn=None):
    run = wandb.init(project=project, name=run_name, config=cfg, reinit=True)
    args = to_namespace(wandb.config)

    Xtr, Ytr, Xva, Yva, Xte, Yte = load_dataset(dataset_name)
    model = NeuralNetwork(args)

    best_val_f1 = -1.0
    for epoch in range(args.epochs):
        model.train(Xtr, Ytr, epochs=1, batch_size=args.batch_size, X_val=Xva, y_val=Yva, verbose=False)
        tr = evaluate_split(model, Xtr, Ytr)
        va = evaluate_split(model, Xva, Yva)
        te = evaluate_split(model, Xte, Yte)

        payload = {
            'epoch': epoch + 1,
            'train_loss': tr['loss'], 'train_acc': tr['acc'], 'train_f1': tr['f1'],
            'val_loss': va['loss'], 'val_acc': va['acc'], 'val_f1': va['f1'],
            'test_loss': te['loss'], 'test_acc': te['acc'], 'test_f1': te['f1'],
            'test_precision': te['precision'], 'test_recall': te['recall'],
        }
        if extras_fn is not None:
            payload.update(extras_fn(model, Xva, Yva))
        wandb.log(payload)

        if va['f1'] > best_val_f1:
            best_val_f1 = va['f1']

    wandb.summary['best_val_f1'] = float(best_val_f1)
    wandb.finish()


In [3]:
PROJECT = 'q2_2_sweep_mnist'
SWEEP_CONFIG = {
    'method': 'bayes',
    'metric': {'name': 'val_f1', 'goal': 'maximize'},
    'early_terminate': {'type': 'hyperband', 'min_iter': 4, 'eta': 3},
    'parameters': {
        'optimizer': {'values': ['nag', 'rmsprop', 'momentum']},
        'learning_rate': {'distribution': 'log_uniform_values', 'min': 1e-4, 'max': 6e-2},
        'batch_size': {'values': [32, 64, 128]},
        'hidden_size': {'values': [[128, 64], [128, 128], [128, 128, 64], [128, 128, 32], [128, 128, 128]]},
        'activation': {'values': ['relu', 'tanh', 'sigmoid']},
        'weight_decay': {'distribution': 'log_uniform_values', 'min': 1e-6, 'max': 1e-3},
        'weight_init': {'value': 'xavier'},
        'loss': {'values': ['cross_entropy', 'mse']},
        'epochs': {'value': 20},
    }
}

def train_with_wandb_for_sweep():
    run = wandb.init(project=PROJECT)
    cfg = dict(wandb.config)
    wandb.finish()
    train_one_run(cfg, project=PROJECT, run_name='sweep_mnist', dataset_name='mnist')

sweep_id = wandb.sweep(SWEEP_CONFIG, project=PROJECT)
wandb.agent(sweep_id, train_with_wandb_for_sweep, count=100)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\91701\_netrc.


Create sweep with ID: 5qihwsi6
Sweep URL: https://wandb.ai/anandhakrishnanm21-indian-institute-of-technology-madras/q2_2_sweep_mnist/sweeps/5qihwsi6


wandb: Agent Starting Run: f6y0vo52 with config:
wandb: 	activation: relu
wandb: 	batch_size: 128
wandb: 	epochs: 20
wandb: 	hidden_size: [128, 128]
wandb: 	learning_rate: 0.00030970282302199184
wandb: 	loss: mse
wandb: 	optimizer: momentum
wandb: 	weight_decay: 0.00012767373915747253
wandb: 	weight_init: xavier
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\91701\_netrc.
wandb: Currently logged in as: anandhakrishnanm21 (anandhakrishnanm21-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
test_acc,▁▃▄▅▆▆▇▇▇▇██████████
test_f1,▁▂▃▄▅▆▆▆▇▇▇█████████
test_loss,██▇▆▅▄▃▃▃▂▂▂▂▁▁▁▁▁▁▁
test_precision,▁▃▄▅▅▆▆▇▇▇██████████
test_recall,▁▃▄▅▅▆▇▇▇▇▇█████████
train_acc,▁▃▄▅▆▆▇▇▇▇▇█████████
train_f1,▁▂▃▄▅▆▆▆▇▇▇█████████
train_loss,██▇▆▅▄▄▃▃▂▂▂▂▁▁▁▁▁▁▁
val_acc,▁▃▄▄▆▆▇▇▇▇▇█████████
+2,...


...

wandb: Agent Starting Run: mamrryak with config:
wandb: 	activation: tanh
wandb: 	batch_size: 32
wandb: 	epochs: 20
wandb: 	hidden_size: [128, 64]
wandb: 	learning_rate: 0.004820590353616726
wandb: 	loss: cross_entropy
wandb: 	optimizer: rmsprop
wandb: 	weight_decay: 0.0005925101650519671
wandb: 	weight_init: xavier
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\91701\_netrc.


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
test_acc,▅▁▅█▇▇▇█▄▆▃▇▆▅▆▆▃▅▇▇
test_f1,▅▁▅█▇▇▇█▄▆▃▇▆▆▆▆▃▅▇▇
test_loss,▅█▅▁▁▂▃▁▆▄▆▂▃▃▃▄▇▅▂▁
test_precision,▄▁▄█▇▇▆█▄▅▂▇▅▅▆▆▃▄▆█
test_recall,▅▁▅█▇▇▇█▄▆▃▇▆▅▆▆▃▅▇▇
train_acc,▅▁▄██▇▇█▄▅▃▇▆▆▇▆▄▅▆█
train_f1,▅▁▅██▇▇█▄▅▃▇▆▆▇▆▄▅▆█
train_loss,▅█▅▁▁▂▃▁▆▄▆▂▃▃▂▃▆▄▃▁
val_acc,▅▁▄▇▇▇▇▇▄▅▄▇▆▆▆▆▄▅▆█
+2,...


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: mke9ydjs with config:
wandb: 	activation: tanh
wandb: 	batch_size: 64
wandb: 	epochs: 20
wandb: 	hidden_size: [128, 64]
wandb: 	learning_rate: 0.0003705850220424403
wandb: 	loss: mse
wandb: 	optimizer: nag
wandb: 	weight_decay: 0.0006519146507916125
wandb: 	weight_init: xavier
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\91701\_netrc.


Traceback (most recent call last):
  File "d:\DL\DL_assignment_1\.dl1env\Lib\site-packages\wandb\agents\pyagent.py", line 304, in _run_job
    self._function()
  File "C:\Users\91701\AppData\Local\Temp\ipykernel_22828\781292715.py", line 23, in train_with_wandb_for_sweep
    train_one_run(cfg, project=PROJECT, run_name='sweep_mnist', dataset_name='mnist')
  File "C:\Users\91701\AppData\Local\Temp\ipykernel_22828\1981384485.py", line 45, in train_one_run
    model.train(Xtr, Ytr, epochs=1, batch_size=args.batch_size, X_val=Xva, y_val=Yva, verbose=False)
  File "D:\DL\DL_assignment_1\src\ann\neural_network.py", line 202, in train
    self.optimizer.lookahead(self._all_params())
  File "D:\DL\DL_assignment_1\src\ann\optimizers.py", line 45, in lookahead
    param -= self.momentum * self.velocity[key]
    ^^^^^
Exception



epoch,▁▂▂▃▃▄▄▅▅▆▆▇▇█
test_acc,▁▅▆▇▇▇████████
test_f1,▁▅▇▇▇█████████
test_loss,█▅▃▃▂▂▂▂▁▁▁▁▁▁
test_precision,▁▅▆▇▇▇████████
test_recall,▁▅▆▇▇▇████████
train_acc,▁▅▆▇▇▇████████
train_f1,▁▅▆▇▇▇████████
train_loss,█▅▄▃▂▂▂▂▁▁▁▁▁▁
val_acc,▁▅▆▇▇▇████████
+2,...


In [4]:
api = wandb.Api()
runs = api.runs(f'{ENTITY}/{PROJECT}')
rows = []
for r in runs:
    if r.summary.get('best_val_f1') is not None:
        rows.append((r.name, r.id, r.summary.get('best_val_f1'), r.summary.get('test_f1')))
rows = sorted(rows, key=lambda x: (x[2] if x[2] is not None else -1), reverse=True)
print('Top runs by best_val_f1:')
for row in rows[:10]:
    print(row)


Top runs by best_val_f1:
('sweep_mnist', 'l3vzy521', 0.9824046992551424, 0.9816882209204651)
('sweep_mnist', 'iimt6p3f', 0.9799073244584408, 0.9794297240471256)
('sweep_mnist', '9yxlt5q2', 0.97802154011939, 0.9718105053285802)
('sweep_mnist', '61b9wqkw', 0.9778963974874244, 0.9771317340747574)
('sweep_mnist', '0zg0fabj', 0.9776621710209934, 0.9792409934640444)
('sweep_mnist', '5xfohoxq', 0.9774796525015012, 0.9787173761594804)
('sweep_mnist', 'skt5wuy0', 0.9773572738945931, 0.9764220940022312)
('sweep_mnist', 'zofxvq2b', 0.9773094886164706, 0.9777485471231844)
('sweep_mnist', 'la82m8sv', 0.976834427673536, 0.9788532915662034)
('sweep_mnist', '5uvht7mk', 0.9767052391543836, 0.97792457688159)
